# Final Visualisation Project BEMM461

## Table of Links

| Description |  Link |
| -- | -- |
| Reflective blog https://ele.exeter.ac.uk/mod/oublog/view.php?id=3234123re |
| Chosen Data sethttps://www.kaggle.com/datasets/atulanandjha/national-stock-exchange-time-series/dataere |  |


https://www.kaggle.com/datasets/atulanandjha/national-stock-exchange-time-series/dataere

## Table of Contents
1. Introduction
2. Dashboard Project
3. Purpose of the Dashboard
4. Background to the Project
5. Design Process and Evolution
6. User Needs and Task Analysis
7. Decision Making Process
8. Chosen Analytics Methods
9. Tools and Libraries Used
10. Datasets and Data Preparation
11. Domain-specific Insights
12. Overview of Dashboard features
13. Interactive User-experience
14. Reflective Evaluation
15. Conclusion
16. References

## 1. Introduction

The National Stock Exchange of India Ltd. (NSE), founded in 1992 in Mumbai, serves as a foundational element of India’s financial infrastructure and is the largest stock exchange in the nation in terms of turnover. As a demutualized electronic platform, NSE has significantly contributed to the modernisation of trading in India by introducing various innovations, including electronic screen-based trading in 1994, index futures, and internet trading in 2000. These developments have confirmed NSE's role as a pioneer in the Indian stock market.

**For this project**, I have chosen performance of two prominent Indian IT firms—Tata Consultancy Services (TCS) and Infosys—both of which hold significant influence in the Indian IT Sector. Furthermore, it integrates data from the NIFTY_IT_INDEX, an important benchmark index that reflects the overall performance of the Indian IT sector. These data sets provide a foundation for analysing individual stock performance in conjunction with overall sector trends.  
This dataset covers trading activity from January 1, 2015, to December 31, 2015, and is structured into three CSV files as follows:

- **TCS Stock Data- Daily trading information for Tata Consultancy Services.**
- **Infosys Stock Data- Daily trading information for Infosys.**
- **NIFTY_IT_INDEX- Consolidated trading data for the IT sector.**

By studying these datasets, this project seeks to study price trends, trading volumes, and sector movements, leading in the development of a detailed and interactive dashboard. This analysis will help various range o  investors, analysts, and students, offering insights that support meaningful comparisons between individual stocks and the broader IT industry benchmark.


## 2.Dashboard Project

# STOCK MARKET DASHBOARD

### Importing necessary libraries

In [1]:
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
from prophet import Prophet  # For predictive analysis

### Loading and Preparing the data

In [2]:
# Load the datasets
nifty_data = pd.read_csv(r'C:\Users\91942\Downloads\nifty_it_index.csv')
tcs_data = pd.read_csv(r'C:\Users\91942\Downloads\tcs_stock.csv')
infy_data = pd.read_csv(r'C:\Users\91942\Downloads\infy_stock.csv')

# Convert 'Date' to datetime format for consistency
nifty_data['Date'] = pd.to_datetime(nifty_data['Date'])
tcs_data['Date'] = pd.to_datetime(tcs_data['Date'])
infy_data['Date'] = pd.to_datetime(infy_data['Date'])

# Filter only relevant columns for simplicity
nifty_data = nifty_data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
tcs_data = tcs_data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
infy_data = infy_data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]

# Add a column to differentiate datasets
nifty_data['Company'] = 'NIFTY_IT_INDEX'
tcs_data['Company'] = 'TCS'
infy_data['Company'] = 'Infosys'

# Combine the datasets into one
combined_data = pd.concat([nifty_data, tcs_data, infy_data])

combined_data.head(26)

,Date,Open,High,Low,Close,Volume,Company
0,2015-01-01,11214.80,11235.75,11166.35,11215.70,4246150,NIFTY_IT_INDEX
1,2015-01-02,11214.65,11399.10,11214.65,11372.10,10004862,NIFTY_IT_INDEX
2,2015-01-05,11369.35,11433.75,11186.95,11248.55,8858018,NIFTY_IT_INDEX
3,2015-01-06,11186.10,11186.10,10909.00,10959.90,12515739,NIFTY_IT_INDEX
4,2015-01-07,11013.20,11042.35,10889.55,10916.00,10976356,NIFTY_IT_INDEX
5,2015-01-08,11031.15,11058.15,10915.05,11018.15,12975117,NIFTY_IT_INDEX
6,2015-01-09,11058.05,11484.90,10932.20,11399.65,24812224,NIFTY_IT_INDEX
7,2015-01-12,11456.00,11565.85,11378.80,11543.65,16505074,NIFTY_IT_INDEX
8,2015-01-13,11545.25,11546.60,11437.95,11502.80,12511358,NIFTY_IT_INDEX
9,2015-01-14,11561.95,11631.55,11521.00,11614.30,12544558,NIFTY_IT_INDEX


### Initialising the Dash App

In [3]:
app = Dash(__name__, external_stylesheets=[dbc.themes.SANDSTONE])
app.title = "Stock Market Dashboard"

### Defining Predictive Analysis Function

In [4]:
def predict_future(company_data, periods=30):
    # Prepare data for Prophet
    df = company_data[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})

    # Initialize and fit the Prophet model
    model = Prophet()
    model.fit(df)

    # Create a DataFrame for future dates
    future = model.make_future_dataframe(periods=periods)
    
    # Generate predictions
    forecast = model.predict(future)

    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

### Defining Descriptive Analysis Function

In [5]:
def calculate_descriptive_stats(data):
    # Calculate metrics
    mean_close = data['Close'].mean()
    median_close = data['Close'].median()
    std_close = data['Close'].std()
    max_close = data['Close'].max()
    min_close = data['Close'].min()

    mean_volume = data['Volume'].mean()
    total_volume = data['Volume'].sum()

    # Return a dictionary of descriptive stats
    return {
        "Mean Closing Price": f"₹{mean_close:.2f}",
        "Median Closing Price": f"₹{median_close:.2f}",
        "Standard Deviation (Price)": f"₹{std_close:.2f}",
        "Max Closing Price": f"₹{max_close:.2f}",
        "Min Closing Price": f"₹{min_close:.2f}",
        "Mean Trading Volume": f"{mean_volume:,.0f}",
        "Total Trading Volume": f"{total_volume:,.0f}"
    }

### Defining Layout

In [6]:
app.layout = dbc.Container([
    dbc.Row(dbc.Col(html.H1("Stock Market Dashboard", className="text-center text-primary mb-4"), width=12)),

    dbc.Row([
        dbc.Col([
            html.Label("Select Company:"),
            dcc.Dropdown(
                id='company-dropdown',
                options=[
                    {'label': 'NIFTY IT INDEX', 'value': 'NIFTY_IT_INDEX'},
                    {'label': 'TCS', 'value': 'TCS'},
                    {'label': 'Infosys', 'value': 'Infosys'}
                ],
                value=['NIFTY_IT_INDEX'],  # Default selection
                multi=True
            )
        ], width=6),

        dbc.Col([
            html.Label("Select Date Range:"),
            dcc.DatePickerRange(
                id='date-picker',
                start_date=combined_data['Date'].min(),
                end_date=combined_data['Date'].max(),
                display_format='YYYY-MM-DD'
            )
        ], width=6),
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(dcc.Graph(id='price-trend-graph'), width=12)
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(id='heatmap-graph'), width=12)
    ]),

    dbc.Row([
        dbc.Col(html.Div(id='descriptive-stats-div', className="p-4"), width=12)
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(id='volume-bar-chart'), width=12)
    ]),

    dbc.Row([
        dbc.Col([
            html.Label("Enable Prediction:"),
            dcc.Checklist(
                id='prediction-toggle',
                options=[{'label': 'Show Predictions', 'value': 'show'}],
                value=[]
            )
        ], width=6),
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(dcc.Graph(id='prediction-graph'), width=12)
    ]),

    dbc.Row([
        dbc.Col([
            html.Button("Download Filtered Data", id="download-btn", className="btn btn-primary"),
            dcc.Download(id="download-dataframe-csv")
        ], width=12, className="text-center")
    ])
], fluid=True)

### Defining Callbacks for Graph, Heat Maps and Descriptive Stats

In [7]:
@app.callback(
    [Output('price-trend-graph', 'figure'),
     Output('volume-bar-chart', 'figure'),
     Output('heatmap-graph', 'figure'),
     Output('descriptive-stats-div', 'children')],
    [Input('company-dropdown', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_dashboard(selected_companies, start_date, end_date):
    filtered_data = combined_data[
        (combined_data['Company'].isin(selected_companies)) &
        (combined_data['Date'] >= start_date) &
        (combined_data['Date'] <= end_date)
    ]

    # Price Trend Chart
    line_chart = px.line(filtered_data, x='Date', y='Close', color='Company', title="Price Trends", template='plotly_dark')

    # Volume Bar Chart
    bar_chart = px.bar(filtered_data, x='Date', y='Volume', color='Company', title="Trading Volume", template='plotly_dark')

    # Heatmap
    heatmap = px.density_heatmap(filtered_data, x='Date', y='Company', z='Volume', title="Heatmap of Trading Volume", template='plotly_dark')

    # Descriptive Stats
    stats = calculate_descriptive_stats(filtered_data)
    stats_elements = [html.H5("Descriptive Analysis:"), html.Ul([html.Li(f"{k}: {v}") for k, v in stats.items()])]

    return line_chart, bar_chart, heatmap, stats_elements

### Defining callback for predictions

In [8]:
@app.callback(
    Output('prediction-graph', 'figure'),
    [Input('company-dropdown', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date'),
     Input('prediction-toggle', 'value')]
)
def update_predictions(selected_companies, start_date, end_date, prediction_toggle):
    if not prediction_toggle:  # If predictions are disabled
        return {}

    # Ensure only one company is selected for predictions
    if len(selected_companies) > 1:
        return px.scatter(title="Please select only one company for predictions.")

    # Filter data for the selected company and date range
    company = selected_companies[0]
    filtered_data = combined_data[
        (combined_data['Company'] == company) &
        (combined_data['Date'] >= start_date) &
        (combined_data['Date'] <= end_date)
    ]

    # Generate predictions
    forecast = predict_future(filtered_data)

    # Create a figure with historical and predicted data
    fig = px.line(filtered_data, x='Date', y='Close', title=f"Predictions for {company}")
    fig.add_scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines', name='Predicted Price')
    fig.add_scatter(x=forecast['ds'], y=forecast['yhat_lower'], mode='lines', name='Lower Bound', line=dict(dash='dot'))
    fig.add_scatter(x=forecast['ds'], y=forecast['yhat_upper'], mode='lines', name='Upper Bound', line=dict(dash='dot'))

    return fig


### Running the App

## Please select Run all cells to get the Dashboard

In [9]:
if __name__ == '__main__':
    app.run(debug=True)  

## 3.Purpose of the Dashboard

The primary purpose of this dashboard is to provide an interactive and user-friendly platform for analysing stock market trends in the Indian IT sector. By using descriptive and predictive analytics, this dashboard aims to support data-driven decision-making for investors, analysts, research students interested in tracking the performance of key IT companies like TCS and Infosys, as well as the broader industry represented by the NIFTY_IT_INDEX. This dashboard allows users to explore historical stock data, such as opening and closing prices, trading volume, and high/low values. It also provides statistical summaries (mean, median, max, min, and standard deviation) to enhance users' understanding of stock performance. By using tools like Prophet, it offers users the ability to forecast future stock trends, enabling better planning and investment strategies. Interactive features like dropdowns, date range pickers, and toggles allow users to customise their analysis and filter data to meet their specific needs.
This dashboard serves a diverse audience, including individual investors, analysts, students, and corporate decision-makers. Investors can track trends and predict future movements, while analysts can assess trading volumes and sectoral patterns. For students and researchers, this dashboard provides a practical example of using data visualization and machine learning to extract insights from real-world datasets. The interactive experience, supported by line charts, bar charts, and heatmaps, ensures the dashboard is both educational and impactful for its users.

## 4. Background to the Project

The Indian stock market plays a vital role in driving economic growth and creating wealth. It is governed by two major stock exchanges—the National Stock Exchange of India (NSE) and the Bombay Stock Exchange (BSE). Since its establishment in 1992, the NSE has been a pioneer in transforming India’s financial landscape, introducing innovations like electronic screen-based trading in 1994 and internet-based trading in 2000. The Indian government has emphasised the IT and telecom sector by allocating US$13.98 billion in the 2024-25 Union Budget, reflecting its growing economic significance.
This dashboard focuses on analysing India's IT sector which is a key pillar of the economy that contributes 7.4% to GDP as of 2022. 

To achieve this, I utilised three key datasets:

- NIFTY_IT_INDEX: A sector-wide index that tracks the collective performance of India’s leading IT companies. This dataset includes daily records of Date, Open, High, Low, Close, and Volume for the year 2015, enabling sector-wide trend analysis.
- TCS: As India’s largest IT services provider, TCS plays a critical role in shaping the broader sector's movements. This dataset contains daily stock data, allowing for analysis of TCS's performance in relation to sectoral trends.
- Infosys: Another key player in India’s IT revolution, Infosys is a major contributor to the industry. This dataset follows the same structure as TCS, with daily records of stock metrics for the year 2015, providing context for comparative analysis with TCS and the NIFTY_IT_INDEX.

The design process was driven by user needs and focused on creating an intuitive, interactive, and insightful dashboard. From the beginning, I identified the needs of key users—investors, analysts, and students, and considered how they would want to interact with the data. To make the dashboard more powerful, I also integrated predictive analysis using Prophet, which allows users to forecast stock prices and better understand future trends. Drawing from the design philosophies of Tufte, Few, and Knaflic, I focused on creating a clean, user-friendly, and interactive experience that "simplifies complexity" while still offering valuable insights.

## 5. Design Process and Evolution

The design process for this project was an iterative journey, going through several stages and following principles in Tufte's and Few's works on visualisation design. I began the project by first brainstorming the features which I want to include include in my dashboard by considering the needs of a user which includes investors, analysts, and students, who would require both historical and predictive insights from the data so I decided to include price trends, trading volumes, predictive analysis, and heatmaps to "simplify complexity" by transforming raw stock data into intuitive visuals. (Cole Nussbaumer Knaflic's)

Initially I created few rough sketches for the dashboard on paper which aimed to display essential stock perfomance metrics using clear and concise visualisations. The first version of the dashboard was developed using Dash and Plotly for visualisation and Pandas for data preparation which included basic line and bar charts for descriptive analysis but lacked interaction. Few drawbacks were noticed at this stage such as static visuals without filtering options and limited ability to handle large datasets. Hence, to address these limitations many improvements were added guided by principles from Tamara Munzner's (Visualisation Analysis & Design,2015) such as dropdowns, date pickers, and toggles for dynamic filtering which enables users to compare stock trends and customise the time range. Then using Prophet, added a forecasting model to visualise future price movements and a heatmap to analyse correlations between volume and price movements. Furthermore, to enhance the dashboard's appeal used UI enhancers such as Dash Bootstrap theme of SANDSTONE. (Cairo's The Functional Art, 2013)

## 6. User Needs and Task Analysis

The primary users include investors, stock analysts, corporate decision-makers, students and researchers, each with a distinct needs: 

1. Investors will need a clear visualisation to monitor stock trends, identify opportunities, and assessing risks in IT sector. Features like price trends, volume charts, and predictive analysis will help them make informed investment decisions.
2. Stock Analysts will require tools to explore patterns, anomalies, and correlations between trading volume and price movements. They will get flexibility to filter data dynamically for in-depth analysis.
3. Corporate Decision Makers will need benchmarking tools to compare company performances (e.g. TCS vs. Infosys) against the broader sector benchmark (NIFTY_IT_INDEX) which this dashboard fulfills by giving the filtering data options.
4. Students and Researchers will seek educational value in understanding stock market behavior, visualisation techniques, and predictive modelling.
5. Data Scientists will require real-world application of tools like Dash, Plotly, and Prophet to explore descriptive and predictive analysis.

By following, Tamara Munzner's (Visualisation Analysis & Design,2015) this dashboard enables users to explore and filter data by using dropdowns and date filters to explore price trends and trading volumes, comparing specific companies and the broader sector dynamically. All features in the dashboard aligns with the distinct needs of users and is displayed with clarity and simplicity to keep the visualisation clean, with intuitive controls (Few's Now You See It, 2009).

## 7. Decision Making Process

The decision-making process for the design and development of this dashboard was guided by keeping in mind the user requirements. First, the choice/selection of datasets (NIFTY_IT_INDEX, TCS, and Infosys) was driven by their relevance to Indian IT Sector. These datasets offers a balance mix of individual company performance and broader sectoral trends which enables meaningful analysis and comparisons. Hence I selected this data which is relevant to the context ensuring insights are actionable.
For Visual Design Choices, guided by Tufte's principles of clarity and simplicity, I decided time-series line charts for analysing historical price trends as they are ideal for showing changes over time, heatmaps to identify correlations between price movements and trading volumes and bar charts to visualise more precisely on trading volumes that allow users to detect spikes and periods of high market activity. The choice of colors and interactivity ensures the dashboard to remain engaging while avoiding clutter (Cairo's The Functional Art, 2013). Furthermore, to integrate interactivity and predictive analysis addition of dropdown menus and date filters are implemented that allow users to filter data dynamically, catering to various analytical needs. The decision to use Prophet was driven by its reliability for time series forecasting which provides forward-looking insights that are particularly valuable to investors and analysts. I tested the dashboard for its functionality, performance, and usuability to ensure all features are working seamlessly (Munzner's Visualisation Analysis & Design, 2015). Lastly, by combining user-centric design with analytical methods each decision contributed to a dashboard that is visually clear, interactive, and insightful.

## 8. Chosen Analytics Methods

This dashboard has a combination of two analytics methods namely descriptive and predictive analysis to uncover insights from the stock datasets for TCS, Infosys and the NIFTY_IT_INDEX. Descriptive analytics is used to summarise and explore the historical trends in stock data and provides a statistical summary of the key metrics for the datasets, focusing on closing prices and trading volumes that will help users understand trends and variability in the stock data. Key statistics include Mean closing price, Median closing price, Standard deviation, Maximum closing price, Minimum closing price, Mean trading volume and total trading volume, by following Cairo's philosophy from The Functional Art (2013), I ensured that no irrelevant metrics are presented and only those that add value to user decision-making are included. Predictive analytics is implemented using Facebook's Prophet model to forecast future stock prices. Prophet is particularly well-suited for time series data due to its ability to handle trends and seasonability. Users can visualise forecasted price trends alongside historical data, enabling forward-looking decision-making.

## 9. Tools and Libraries Used

The development of this dashboard required the use of a variety of data visualisation, analysis, and development libraries. The choice of tools was guided by the principles of clarity, simplicity, and interactivity, by Edward Tufte and Stephen Few in their works on effective visualization. Each tool played a critical role in enhancing user experience, interactivity, and analytical capabilities. Dash served as the core frameword for building this interactive dashboard. It creates a fully interactive, dynamic dashboard with minimal front-end coding and its layout prioritises clarity and usability allowing the integration of dropdowns, date filters and dynamic charts (Stephen Few). Plotly is used to create line charts, heatmaps, and bar chart as it offers visually appealing and engaging charts. Visual elements are designed with features like tooltips and hover interactions to improve user engagement. Pandas library is used to clean data, manipulating and data preparation. It is efficient in handling large datasets for fast filtering and aggregation which is a core requirement identified in Munzner's Visualisation Analysis and Design (2015). Lastly, Prophet is used for predictive analysis to forecast future stock prices as its time-series forecasting capabilities align with the need for forward-looking insights and it is integrated as part of the dashboard's interactive toggle.

## 10. Datasets and Data Preparation

The project uses three key datasets—NIFTY_IT_INDEX, TCS, and Infosys, to analyse the Indian IT sector's stock performance. These datasets are essential for both descriptive analysis and predictive forecasting. The preparation of data is guided by best practices from Tamara Munzner and Stephen Few, ensuring clarity, accuracy, and usability. NIFTY_IT_INDEX is a sector-wide benchmark tracking IT company performance, stock data for TCS (Tata Consultancy Services), a leading IT firm and Infosys stock data which is one of India's largest IT firms. All datasets include Date, Open, High, Low, Close, and Volume columns for daily trading data from January 1, 2015, to December 31, 2015. This period allows for meaningful analysis of market patterns. Cleaned the data by handling all the missing/null values, ensuring consistency in the date format across datasets which reflects Tufte's emphasis on "data integrity" in The Visual Display of Quantitative Information (1983). Transformed the data by combining TCS, Infosys, and NIFTY_IT_INDEX datasets using Pandas and adding a company column to distinguish data by source for comparative analysis. Selected only essential columns such as date, open, high, low, close, volume reducing data-ink ration (Few's principle, 2009) ensuring data cleanliness and consistency.

## 11. Domain-specific Insights

The Indian IT sector plays a vital role in the country’s economy, contributing 7.4% to India’s GDP as of 2022. Major players like TCS and Infosys are key contributors to this growth, and their stock performance is closely watched by investors, analysts, and corporate decision-makers. The NIFTY_IT_INDEX serves as a sectoral benchmark, providing a comparative lens to assess the performance of individual IT firms against the broader industry. 
This dashboard offers a platform to explore these insights, providing users with critical information on price movements, trading volume trends, and future forecasts. Through descriptive analysis, it was observed that the NIFTY_IT_INDEX displayed relatively low volatility with a standard deviation of ₹466.68, signaling market stability. The price peak of ₹12,855.90 and a low of ₹10,798.25 revealed shifts in sectoral performance. Additionally, the total trading volume of 3.42 billion reflects strong market activity and liquidity, indicating high investor interest. Spikes in trading volume were often linked to company announcements and sectoral news, supporting Few’s assertion that users seek patterns in financial data.  
Users of this dashboard include investors, analysts, corporate decision-makers, and students. Their key tasks include tracking stock trends, comparing stock performance (TCS, Infosys, and NIFTY_IT_INDEX), and using predictive analytics to forecast future price movements. To meet these needs, the dashboard offers interactive line charts, bar charts, and heatmaps, allowing users to explore trends, assess performance, and visualise data correlations. Predictive analysis using Prophet further empowers users to project future stock prices, supporting strategic investment decisions. This aligns with Munzner's concept of "what-if" analysis.  
To ensure reliability and address potential threats to validity, measures were taken to reduce issues such as data bias, model inaccuracies, and technical errors. Data was cleaned and formatted for consistency, and extensive testing was conducted to ensure model accuracy and dashboard interactivity. Forecasting models were validated by comparing predicted values against actual results to ensure robustness.  

## 12. Overview of Dashboard features

The dashboard provides an interactive platform to visualise and analyse the performance of TCS, Infosys,  NIFTY_IT_INDEX. It combines descriptive analytics, predictive forecasts and user interactivity to cater to the needs of investors, analysts and business decision makers. The design adheres to Stephen Few's principles of simplicity and clarity to ensure ease of use.

Key Features are as follows:
1. Line Charts for price trends to analyse historical highs, lows, and movements over time.
2. Bar Charts for trading volume, enabling users to detect spikes in activity.
3. Heatmaps for volume-price correlation, revealing hidden patterns in market behavior.
4. Dropdowns for company selection (TCS, Infosys, or NIFTY_IT_INDEX).
5. Date Pickers to customise time periods for analysis.
6. Toggle Switch to enable Prophet-based price forecasts for future trend predictions.
7. Descriptive statistics (mean, median, min, max) offer quick insights on stock performance.
8. Predictive analysis shows future price forecasts, aiding in forward-looking decision-making.
9. Allows you to download your filtered data.

This feature-rich dashboard fulfills the distinct needs of various users efficiently.


## 13. Interactive User-experience

This dashboard offers an intuitive and seamless interactive user experience through the use of dynamic controls such as dropdown menus, date filters, and toggle switches, allowing users to tailor their analysis to specific needs. Dropdown menus enable users to select a specific company (TCS, Infosys, or NIFTY_IT_INDEX), while date filters allow users to filter data for a chosen time range, offering a more personalised exploration of stock trends. Users can activate Prophet-based predictive analysis via a toggle switch, that enables them to visualise future price forecasts alongside historical data. Interactivity extends to the visualisations themselves, with hover effects that display precise data points for closing prices and trading volumes. The charts—line, bar, and heatmap visualisations, update dynamically based on user input, which ensures flexibility and responsiveness. Drawing from Stephen Few’s principles of simplicity and usability, these features enhance user engagement and provides a smooth, efficient experience. This dashboard also aligns with Munzner's concept of supporting "what-if" analysis, enabling users to test different scenarios. By prioritising interactivity, the dashboard ensures a user-centric experience, allowing investors, analysts, and decision-makers to explore stock trends, compare performance, and gain forward-looking insights with ease.

## 14. Reflective Evaluation

The development of this dashboard followed an iterative design process, allowing for continuous improvement and refinement. Tamara Munzner’s principles from Visualisation Analysis & Design (2015) emphasised the importance of iterative testing and user feedback, which guided the evolution of key features.

Strengths:
- User Interactivity: Dropdowns, date pickers, and toggles enable users to explore company-specific data and adjust the time frame, aligning with Few’s focus on usability.
- Predictive Analysis: The integration of Prophet forecasting empowers users with future insights, making the dashboard forward-looking and actionable.
- Clarity of Visuals: Following Tufte's principles, charts were designed with simplicity in mind, ensuring users can quickly interpret key information.

Challenges:
- Data Cleaning: Ensuring consistent date formats and handling missing values was time-consuming.
- Performance Optimisation: Managing large datasets required efficient callbacks to avoid lag in chart rendering.

Lessons Learned:
- Iterative testing and user feedback played a crucial role in improving interactivity and optimising dashboard performance.
- Balancing clarity and interactivity required strategic design choices, ensuring visuals remained informative yet simple.

## 15. Conclusion

The development of this interactive stock market dashboard has been a rewarding journey, resulting in a tool that empowers users to explore, analyse, and predict stock trends for TCS, Infosys, and the NIFTY_IT_INDEX. By integrating descriptive analytics (mean, median, and standard deviation) with predictive forecasting using Prophet, this dashboard provides both a snapshot of past performance and a glimpse into future trends. This combination serves the needs of investors, analysts, and students looking for actionable insights.  
Interactive features like dropdown menus, date pickers, and toggle switches allow users to personalise their analysis, making it easy to visualise specific companies or explore specific time periods. The use of line charts, bar charts, and heatmaps offers a clear, visual approach to understanding stock movements. The project’s design was guided by the principles of clarity, simplicity, and interactivity, drawing from the ideas of Few, Munzner, and Tufte.  
While challenges such as data cleaning and optimising dashboard performance were encountered, they were addressed through thoughtful design choices and iterative testing. Ultimately, the dashboard provides users with an engaging, easy-to-use platform for understanding stock market trends. It successfully achieves its goal of supporting better investment decisions and fostering a deeper understanding of financial data analysis.

## 16. References

1. Abras, C., Maloney-Krichmar, D., & Preece, J. (2004). User-centered design. In W. S. Bainbridge (Ed.), Encyclopedia of Human-Computer Interaction (pp. 445–456). Thousand Oaks, CA: Sage Publications.
2. Cairo, A. (2013). The functional art: An introduction to information graphics and visualisation. New Riders.
3. Few, S. (2004). Show me the numbers: Designing tables and graphs to enlighten. Analytics Press.
Chapter 9: Visual Perception and Quantitative Data
Chapter 10: Designing Tables and Graphs
Chapter 11: Graph Design Principles
4. Few, S. (2009). Now you see it: Simple visualisation techniques for quantitative analysis. Analytics Press.
5. Forbes Advisor. (2023). The best data visualisation tools of 2023. Forbes. Retrieved from https://www.forbes.com/advisor/business/software/best-data-visualisation-tools/
6. Knaflic, C. N. (2015). Storytelling with data: A data visualisation guide for business professionals. Wiley.
7. Munzner, T. (2015). Visualisation analysis & design. CRC Press.
8. Chapter 4: Data Abstraction in Visualisation Design
Prophet. (n.d.). Prophet Library. Retrieved from https://facebook.github.io/prophet/
9. Segel, E., & Heer, J. (2010). Narrative visualisation: Telling stories with data. IEEE Transactions on Visualisation and Computer Graphics, 16(6), 1139–1148. https://doi.org/10.1109/TVCG.2010.179
10. Tufte, E. (1983). The visual display of quantitative information. Graphics Press.
11. National Stock Exchange of India. (n.d.). About NSE. Retrieved from https://www.nseindia.com/
12. India Brand Equity Foundation (IBEF). (2024). IT & BPM industry in India. Retrieved from https://www.ibef.org/industry/it-bpo-india
13. Ministry of Finance, Government of India. (2024). Union Budget 2024-25: Key highlights. Retrieved from https://www.indiabudget.gov.in/
14. India Brand Equity Foundation (IBEF). (2022). Indian IT & BPM industry report. Retrieved from https://www.ibef.org/industry/it-bpo-india
15. India Brand Equity Foundation (IBEF). (2022). Indian IT & BPM industry report. Retrieved from https://www.ibef.org/industry/it-bpo-india

In [10]:
import os

# Step 1: Get Jupyter configuration directory
jupyter_path = os.path.join(os.path.expanduser('~'), '.jupyter', 'custom')
os.makedirs(jupyter_path, exist_ok=True)

# Step 2: Create the custom.css file
css_path = os.path.join(jupyter_path, 'custom.css')

# Step 3: Write the custom CSS styles to the file
css_content = """
/* Custom CSS for Jupyter Notebook */
body {
    font-family: Arial !important;
    font-size: 12pt !important;
    line-height: 1.5 !important;
}
"""

with open(css_path, 'w') as f:
    f.write(css_content)

print(f"Custom CSS file created at: {css_path}")


Custom CSS file created at: C:\Users\91942\.jupyter\custom\custom.css
